In [ ]:
import pandas as pd
import numpy as np
import altair as alt 

In [ ]:
photos_df = pd.read_csv('../data/congress_members_with_photos.csv')

In [ ]:
photos_df.head()

In [ ]:
df1_with_images = photos_df.copy()

# Compute average bills per member for each generation
avg_bills_by_gen = (
    df1_with_images.groupby('Generation', as_index=False)['BillCount']
    .mean()
    .rename(columns={'BillCount': 'AvgBillsPerMember'})
)

# Merge the average back into the main dataframe so it appears in tooltips
df1_with_images = df1_with_images.merge(avg_bills_by_gen, on='Generation', how='left')

# Individual member activity chart WITH image tooltips

selection = alt.selection_point(fields=['Generation'], bind='legend')

member_activity_with_images = alt.Chart(df1_with_images).mark_circle(opacity=0.7).add_params(
    selection
).encode(
    x=alt.X('BirthYear:Q', title='Birth Year', scale=alt.Scale(domain=[1930, 2000]), axis=alt.Axis(format='d')),
    y=alt.Y('BillCount:Q', title='Number of Bills Sponsored'),
    color=alt.Color('Generation:N', scale=alt.Scale(scheme='category10')),
    opacity=alt.when(selection).then(alt.value(.8)).otherwise(alt.value(0.2)),
    size=alt.value(60),
    tooltip=[
        alt.Tooltip('PhotoURL:N', title=' '),  # Photo appears first in tooltip
        'Name:N',
        'Generation:N', 
        'Party:N', 
        alt.Tooltip('BirthYear:Q', format='d'),  # 'd' format removes comma
        'BillCount:Q',
        alt.Tooltip('AvgBillsPerMember:Q', title='Gen Avg Bills', format='.1f')
    ]
).properties(
    title='Individual Member Activity by Birth Year',
    width='container',
    height='container'
)

print("Sample of data with image URLs:")
print(df1_with_images[['Name', 'Party', 'PhotoURL', 'AvgBillsPerMember']].head(3))
member_activity_with_images


In [ ]:
# Save the chart as an HTML file
member_activity_with_images.save('../visualizations/member_activity_scatter.html')

In [ ]:
percentage_df = pd.DataFrame({
    'Generation': ['Baby Boomer', 'Generation X', 'Millennial', 'Silent Generation', 'Generation Z'],
    'Percentage': [42.64, 39.11, 13.59, 4.47, 0.19]
})

population_percentage_df = pd.DataFrame({
    'Generation': ['Baby Boomer', 'Generation X', 'Millennial', 'Silent Generation', 'Generation Z'],
    'Percentage': [19.67, 19.27, 21.81, 4.48, 20.81]
})

merged_df = pd.merge(percentage_df, population_percentage_df, on='Generation', suffixes=('_Congress', '_Population'))

In [ ]:
merged_df.head()

In [ ]:
merged_df = merged_df.rename(columns={
    'Percentage_Congress': 'Congress',
    'Percentage_Population': 'US Population'})

In [ ]:
melted_df = merged_df.melt(id_vars='Generation', value_vars=['Congress', 'US Population'],
                          var_name='Percentage_Type', value_name='Percentage')
melted_df.head(10)

In [ ]:
sorted_generations = ['Baby Boomer', 'Generation X', 'Millennial', 'Silent Generation', 'Generation Z']

In [ ]:
input_dropdown = alt.binding_select(options=['Congress', 'US Population'], name='Percentage of: ')
selection = alt.selection_point(fields=['Percentage_Type'], bind=input_dropdown, init={'Percentage_Type': 'Congress'}) 
color = alt.condition(selection, alt.Color('Percentage_Type:N', scale=alt.Scale(scheme='category10')), alt.value('lightgray'))
generational_representation = alt.Chart(melted_df).mark_bar().encode(
    x=alt.X('Generation', axis=alt.Axis(labelAngle=45)),
    y=alt.Y('Percentage', axis=alt.Axis(title='Percentage (%)'), scale=alt.Scale(domain=[0, 45])),
    color=color,
    tooltip=['Generation', 'Percentage_Type', 'Percentage']
).properties(
    title='Generational Representation: Congress vs. US Population',
    width='container',
).interactive().add_params(selection)

generational_representation

In [ ]:
generational_representation.save('../visualizations/generation_representation_bar_chart.html')

In [ ]:
df_party = photos_df[photos_df['Generation'] != 'Unknown'].groupby(['Generation', 'Party']).size().reset_index(name='Count')

# Map party codes to full names
party_mapping = {'D': 'Democrat', 'R': 'Republican', 'I': 'Independent'}
df_party['Party'] = df_party['Party'].map(party_mapping)

print(df_party)

In [ ]:
generation_viz = alt.Chart(df_party).mark_bar(size=40).encode(
        y=alt.Y('Generation:N', sort=alt.EncodingSortField(field='Count', op='sum', order='descending')).axis(alt.Axis(labelAngle=0)),
        x=alt.X('Count:Q', title='Number of Members'),
        color=alt.Color('Party:N', 
                       scale=alt.Scale(domain=['Democrat', 'Republican', 'Independent'], 
                                     range=['#0015BC', '#FF0000', '#9966CC']),
                       legend=alt.Legend(title='Political Party')),
        tooltip=['Generation', 'Party', 'Count']
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        grid=False
    ).properties(
        title='Congressional Members by Generation and Party',
        width='container',
        height=300,
    )
generation_viz

In [ ]:
generation_viz.save('../visualizations/generation_party_bar_chart.html')